<a href="https://colab.research.google.com/github/nickdiamond913/MIS2800/blob/main/IC4_Diamond.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "id": "063ed243",
      "metadata": {
        "id": "063ed243"
      },
      "source": [
        "# IC (Feb 04) — Text Data in SQL (Student Version)\n",
        "\n",
        "This in-class exercise practices the ideas from the Feb 04 slides:\n",
        "- Text / character data and why it’s tricky\n",
        "- `GROUP BY` + `COUNT()`\n",
        "- `ORDER BY` (frequency vs alphabetical)\n",
        "- Case sensitivity and `LOWER()` / `UPPER()`\n",
        "- Spaces and `TRIM()`\n",
        "- `LIKE` searches (and why they can be surprising)\n",
        "\n",
        "**Instructions:** Run cells top → bottom. Write SQL inside the `query` blocks.\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "4577a25d",
      "metadata": {
        "id": "4577a25d"
      },
      "source": [
        "## 0) Setup (Run)\n",
        "Creates a SQLite database `text_issues.db` and a table `product` with messy text."
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "!pip install --upgrade pandas ipython-sql prettytable==3.10.1"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "qCk_Hwrc3vVy",
        "outputId": "07eda4af-b11d-43eb-bb69-3cc6fcfac67a"
      },
      "id": "qCk_Hwrc3vVy",
      "execution_count": 1,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Requirement already satisfied: pandas in /usr/local/lib/python3.12/dist-packages (3.0.0)\n",
            "Requirement already satisfied: ipython-sql in /usr/local/lib/python3.12/dist-packages (0.5.0)\n",
            "Requirement already satisfied: prettytable==3.10.1 in /usr/local/lib/python3.12/dist-packages (3.10.1)\n",
            "Requirement already satisfied: wcwidth in /usr/local/lib/python3.12/dist-packages (from prettytable==3.10.1) (0.5.0)\n",
            "Requirement already satisfied: numpy>=1.26.0 in /usr/local/lib/python3.12/dist-packages (from pandas) (2.0.2)\n",
            "Requirement already satisfied: python-dateutil>=2.8.2 in /usr/local/lib/python3.12/dist-packages (from pandas) (2.9.0.post0)\n",
            "Requirement already satisfied: ipython in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (7.34.0)\n",
            "Requirement already satisfied: sqlalchemy>=2.0 in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (2.0.46)\n",
            "Requirement already satisfied: sqlparse in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (0.5.5)\n",
            "Requirement already satisfied: six in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (1.17.0)\n",
            "Requirement already satisfied: ipython-genutils in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (0.2.0)\n",
            "Requirement already satisfied: greenlet>=1 in /usr/local/lib/python3.12/dist-packages (from sqlalchemy>=2.0->ipython-sql) (3.3.1)\n",
            "Requirement already satisfied: typing-extensions>=4.6.0 in /usr/local/lib/python3.12/dist-packages (from sqlalchemy>=2.0->ipython-sql) (4.15.0)\n",
            "Requirement already satisfied: setuptools>=18.5 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (75.2.0)\n",
            "Requirement already satisfied: jedi>=0.16 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (0.19.2)\n",
            "Requirement already satisfied: decorator in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (4.4.2)\n",
            "Requirement already satisfied: pickleshare in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (0.7.5)\n",
            "Requirement already satisfied: traitlets>=4.2 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (5.7.1)\n",
            "Requirement already satisfied: prompt-toolkit!=3.0.0,!=3.0.1,<3.1.0,>=2.0.0 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (3.0.52)\n",
            "Requirement already satisfied: pygments in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (2.19.2)\n",
            "Requirement already satisfied: backcall in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (0.2.0)\n",
            "Requirement already satisfied: matplotlib-inline in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (0.2.1)\n",
            "Requirement already satisfied: pexpect>4.3 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (4.9.0)\n",
            "Requirement already satisfied: parso<0.9.0,>=0.8.4 in /usr/local/lib/python3.12/dist-packages (from jedi>=0.16->ipython->ipython-sql) (0.8.5)\n",
            "Requirement already satisfied: ptyprocess>=0.5 in /usr/local/lib/python3.12/dist-packages (from pexpect>4.3->ipython->ipython-sql) (0.7.0)\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# Install the SQL extension (not in sources, but required for magics)\n",
        "!pip install ipython-sql\n",
        "print(\"ipython sql installed\")\n",
        "# lines of code will run below for the installation"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "JA3adGTs38q_",
        "outputId": "a7894ed2-b0ed-4391-974c-8e58e0c48bff"
      },
      "id": "JA3adGTs38q_",
      "execution_count": 2,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "Requirement already satisfied: ipython-sql in /usr/local/lib/python3.12/dist-packages (0.5.0)\n",
            "Requirement already satisfied: prettytable in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (3.10.1)\n",
            "Requirement already satisfied: ipython in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (7.34.0)\n",
            "Requirement already satisfied: sqlalchemy>=2.0 in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (2.0.46)\n",
            "Requirement already satisfied: sqlparse in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (0.5.5)\n",
            "Requirement already satisfied: six in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (1.17.0)\n",
            "Requirement already satisfied: ipython-genutils in /usr/local/lib/python3.12/dist-packages (from ipython-sql) (0.2.0)\n",
            "Requirement already satisfied: greenlet>=1 in /usr/local/lib/python3.12/dist-packages (from sqlalchemy>=2.0->ipython-sql) (3.3.1)\n",
            "Requirement already satisfied: typing-extensions>=4.6.0 in /usr/local/lib/python3.12/dist-packages (from sqlalchemy>=2.0->ipython-sql) (4.15.0)\n",
            "Requirement already satisfied: setuptools>=18.5 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (75.2.0)\n",
            "Requirement already satisfied: jedi>=0.16 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (0.19.2)\n",
            "Requirement already satisfied: decorator in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (4.4.2)\n",
            "Requirement already satisfied: pickleshare in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (0.7.5)\n",
            "Requirement already satisfied: traitlets>=4.2 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (5.7.1)\n",
            "Requirement already satisfied: prompt-toolkit!=3.0.0,!=3.0.1,<3.1.0,>=2.0.0 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (3.0.52)\n",
            "Requirement already satisfied: pygments in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (2.19.2)\n",
            "Requirement already satisfied: backcall in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (0.2.0)\n",
            "Requirement already satisfied: matplotlib-inline in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (0.2.1)\n",
            "Requirement already satisfied: pexpect>4.3 in /usr/local/lib/python3.12/dist-packages (from ipython->ipython-sql) (4.9.0)\n",
            "Requirement already satisfied: wcwidth in /usr/local/lib/python3.12/dist-packages (from prettytable->ipython-sql) (0.5.0)\n",
            "Requirement already satisfied: parso<0.9.0,>=0.8.4 in /usr/local/lib/python3.12/dist-packages (from jedi>=0.16->ipython->ipython-sql) (0.8.5)\n",
            "Requirement already satisfied: ptyprocess>=0.5 in /usr/local/lib/python3.12/dist-packages (from pexpect>4.3->ipython->ipython-sql) (0.7.0)\n",
            "ipython sql installed\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# Load the SQL magic extension\n",
        "%load_ext sql"
      ],
      "metadata": {
        "id": "TvkjJtOv3_8B"
      },
      "id": "TvkjJtOv3_8B",
      "execution_count": 3,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "import sqlite3, pandas as pd"
      ],
      "metadata": {
        "id": "QsSmcfJdAw7C"
      },
      "id": "QsSmcfJdAw7C",
      "execution_count": 6,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# This creates a brand-new, empty database file named 'text_issues.db'\n",
        "conn = sqlite3.connect('text_issues.db')\n",
        "print(\"database created\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "aSn5CsDFAyNj",
        "outputId": "72775aa8-9f76-4902-aa57-548a61c82517"
      },
      "id": "aSn5CsDFAyNj",
      "execution_count": 7,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            "database created\n"
          ]
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "%sql sqlite:///text_issues.db"
      ],
      "metadata": {
        "id": "VUD4IenSBAE_"
      },
      "id": "VUD4IenSBAE_",
      "execution_count": 8,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "# create a table called inventory_table\n",
        "%%sql\n",
        "CREATE TABLE inventory_table (\n",
        "  id INTEGER PRIMARY KEY,\n",
        "  category TEXT,\n",
        "  note TEXT\n",
        ") STRICT;"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "w2rE3JvHBJEg",
        "outputId": "e5c840ba-0d17-419c-b9b3-508ff9a52612"
      },
      "id": "w2rE3JvHBJEg",
      "execution_count": 10,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 10
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# complet the table creation\n",
        "%%sql\n",
        "INSERT INTO inventory_table (id, category, note)\n",
        "VALUES\n",
        "  (1,  \"Apple\",   \"fresh\"),\n",
        "  (2,  \"apple\",   \"fresh\"),\n",
        "  (3,  \"APPLE\",   \"fresh\"),\n",
        "  (4,  \"Banana\",  \"ripe\"),\n",
        "  (5,  \"banana\",  \"ripe\"),\n",
        "  (6,  \"banana \", \"ripe\"),\n",
        "  (7,  \" banana\", \"ripe\"),\n",
        "  (8,  \"to-do\",   \"punctuation\"),\n",
        "  (9,  \"to–do\",   \"punctuation\"),\n",
        "  (10, \"\",        \"empty string\"),\n",
        "  (11, 'None',      \"NULL value\");"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "glFO3uNIB-Ts",
        "outputId": "2960c3d2-804f-4280-a0e8-806d9bc241e0"
      },
      "id": "glFO3uNIB-Ts",
      "execution_count": 14,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "11 rows affected.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 14
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "%%sql\n",
        "SELECT * FROM inventory_table;"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 305
        },
        "id": "p75EbtQyCuvD",
        "outputId": "ca9e5a71-5849-4480-d38d-9fbf632e8a4b"
      },
      "id": "p75EbtQyCuvD",
      "execution_count": 15,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[(1, 'Apple', 'fresh'),\n",
              " (2, 'apple', 'fresh'),\n",
              " (3, 'APPLE', 'fresh'),\n",
              " (4, 'Banana', 'ripe'),\n",
              " (5, 'banana', 'ripe'),\n",
              " (6, 'banana ', 'ripe'),\n",
              " (7, ' banana', 'ripe'),\n",
              " (8, 'to-do', 'punctuation'),\n",
              " (9, 'to–do', 'punctuation'),\n",
              " (10, '', 'empty string'),\n",
              " (11, 'None', 'NULL value')]"
            ],
            "text/html": [
              "<table>\n",
              "    <thead>\n",
              "        <tr>\n",
              "            <th>id</th>\n",
              "            <th>category</th>\n",
              "            <th>note</th>\n",
              "        </tr>\n",
              "    </thead>\n",
              "    <tbody>\n",
              "        <tr>\n",
              "            <td>1</td>\n",
              "            <td>Apple</td>\n",
              "            <td>fresh</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>2</td>\n",
              "            <td>apple</td>\n",
              "            <td>fresh</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>3</td>\n",
              "            <td>APPLE</td>\n",
              "            <td>fresh</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>4</td>\n",
              "            <td>Banana</td>\n",
              "            <td>ripe</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>5</td>\n",
              "            <td>banana</td>\n",
              "            <td>ripe</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>6</td>\n",
              "            <td>banana </td>\n",
              "            <td>ripe</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>7</td>\n",
              "            <td> banana</td>\n",
              "            <td>ripe</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>8</td>\n",
              "            <td>to-do</td>\n",
              "            <td>punctuation</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>9</td>\n",
              "            <td>to–do</td>\n",
              "            <td>punctuation</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>10</td>\n",
              "            <td></td>\n",
              "            <td>empty string</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>11</td>\n",
              "            <td>None</td>\n",
              "            <td>NULL value</td>\n",
              "        </tr>\n",
              "    </tbody>\n",
              "</table>"
            ]
          },
          "metadata": {},
          "execution_count": 15
        }
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# Preview the data\n",
        "run_sql(\"SELECT * FROM inventory_table;\")"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 390
        },
        "id": "jSXRKR0gC5Si",
        "outputId": "2b183602-07db-4f8e-8a3d-d333ed27392a"
      },
      "id": "jSXRKR0gC5Si",
      "execution_count": 26,
      "outputs": [
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "    id category          note\n",
              "0    1    Apple         fresh\n",
              "1    2    apple         fresh\n",
              "2    3    APPLE         fresh\n",
              "3    4   Banana          ripe\n",
              "4    5   banana          ripe\n",
              "5    6  banana           ripe\n",
              "6    7   banana          ripe\n",
              "7    8    to-do   punctuation\n",
              "8    9    to–do   punctuation\n",
              "9   10           empty string\n",
              "10  11     None    NULL value"
            ],
            "text/html": [
              "<div>\n",
              "<style scoped>\n",
              "    .dataframe tbody tr th:only-of-type {\n",
              "        vertical-align: middle;\n",
              "    }\n",
              "\n",
              "    .dataframe tbody tr th {\n",
              "        vertical-align: top;\n",
              "    }\n",
              "\n",
              "    .dataframe thead th {\n",
              "        text-align: right;\n",
              "    }\n",
              "</style>\n",
              "<table border=\"1\" class=\"dataframe\">\n",
              "  <thead>\n",
              "    <tr style=\"text-align: right;\">\n",
              "      <th></th>\n",
              "      <th>id</th>\n",
              "      <th>category</th>\n",
              "      <th>note</th>\n",
              "    </tr>\n",
              "  </thead>\n",
              "  <tbody>\n",
              "    <tr>\n",
              "      <th>0</th>\n",
              "      <td>1</td>\n",
              "      <td>Apple</td>\n",
              "      <td>fresh</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>1</th>\n",
              "      <td>2</td>\n",
              "      <td>apple</td>\n",
              "      <td>fresh</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>2</th>\n",
              "      <td>3</td>\n",
              "      <td>APPLE</td>\n",
              "      <td>fresh</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>3</th>\n",
              "      <td>4</td>\n",
              "      <td>Banana</td>\n",
              "      <td>ripe</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>4</th>\n",
              "      <td>5</td>\n",
              "      <td>banana</td>\n",
              "      <td>ripe</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>5</th>\n",
              "      <td>6</td>\n",
              "      <td>banana</td>\n",
              "      <td>ripe</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>6</th>\n",
              "      <td>7</td>\n",
              "      <td>banana</td>\n",
              "      <td>ripe</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>7</th>\n",
              "      <td>8</td>\n",
              "      <td>to-do</td>\n",
              "      <td>punctuation</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>8</th>\n",
              "      <td>9</td>\n",
              "      <td>to–do</td>\n",
              "      <td>punctuation</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>9</th>\n",
              "      <td>10</td>\n",
              "      <td></td>\n",
              "      <td>empty string</td>\n",
              "    </tr>\n",
              "    <tr>\n",
              "      <th>10</th>\n",
              "      <td>11</td>\n",
              "      <td>None</td>\n",
              "      <td>NULL value</td>\n",
              "    </tr>\n",
              "  </tbody>\n",
              "</table>\n",
              "</div>"
            ]
          },
          "metadata": {},
          "execution_count": 26
        }
      ]
    },
    {
      "cell_type": "markdown",
      "id": "42c1f69c",
      "metadata": {
        "id": "42c1f69c"
      },
      "source": [
        "## Part A — Quick Concept Check (We do it together)\n",
        "1. Which values in `category` refer to the *same* category but look different?\n",
        "2. What is the difference between an **empty string** (`''`) and **NULL**?\n",
        "3. Why might punctuation cause matching problems?\n"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [],
      "metadata": {
        "id": "wrnPGu3srHRb"
      },
      "id": "wrnPGu3srHRb"
    },
    {
      "cell_type": "markdown",
      "id": "b4762014",
      "metadata": {
        "id": "b4762014"
      },
      "source": [
        "## Part B1 — Grouping and counting (raw text)\n",
        "Which product categories have the most products, and how many products are in each category (sorted from most to least)"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "%%sql\n",
        "-- Group by category and count.\n",
        "\n",
        "\n",
        "\n"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 305
        },
        "id": "_wm80f2_4rI0",
        "outputId": "621599d7-9666-4346-bbba-f564d74001ac"
      },
      "id": "_wm80f2_4rI0",
      "execution_count": 43,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[('to–do', 1),\n",
              " ('to-do', 1),\n",
              " ('banana ', 1),\n",
              " ('banana', 1),\n",
              " ('apple', 1),\n",
              " ('None', 1),\n",
              " ('Banana', 1),\n",
              " ('Apple', 1),\n",
              " ('APPLE', 1),\n",
              " (' banana', 1),\n",
              " ('', 1)]"
            ],
            "text/html": [
              "<table>\n",
              "    <thead>\n",
              "        <tr>\n",
              "            <th>category</th>\n",
              "            <th>COUNT(*)</th>\n",
              "        </tr>\n",
              "    </thead>\n",
              "    <tbody>\n",
              "        <tr>\n",
              "            <td>to–do</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>to-do</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>banana </td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>banana</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>apple</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>None</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>Banana</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>Apple</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td>APPLE</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td> banana</td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "        <tr>\n",
              "            <td></td>\n",
              "            <td>1</td>\n",
              "        </tr>\n",
              "    </tbody>\n",
              "</table>"
            ]
          },
          "metadata": {},
          "execution_count": 43
        }
      ]
    },
    {
      "cell_type": "markdown",
      "id": "a208396f",
      "metadata": {
        "id": "a208396f"
      },
      "source": [
        "## Part B2 — Fix grouping with LOWER() - individual one"
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "%%sql\n",
        "-- Make category case-insensitive: group by LOWER(category) and count (ignore NULLs)\n"
      ],
      "metadata": {
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "id": "ZIQjh7HAH_zP",
        "outputId": "d07cafe2-6d4b-492d-a85a-e027ae2658e0"
      },
      "id": "ZIQjh7HAH_zP",
      "execution_count": 34,
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 34
        }
      ]
    },
    {
      "cell_type": "markdown",
      "id": "727f6c7e",
      "metadata": {
        "id": "727f6c7e"
      },
      "source": [
        "## Part B3 — Fix spaces with TRIM() + LOWER()"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": 35,
      "id": "dd65473d",
      "metadata": {
        "id": "dd65473d",
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "outputId": "ee6bbf79-6870-48c2-8623-a7717428cf0e"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 35
        }
      ],
      "source": [
        "%%sql\n",
        "\n",
        "-- Clean spaces and case: group by LOWER(TRIM(category)) and count (ignore NULLs).\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "7213e247",
      "metadata": {
        "id": "7213e247"
      },
      "source": [
        "## Part C1 — Order by category (alphabetical)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": 36,
      "id": "ae34c82b",
      "metadata": {
        "id": "ae34c82b",
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "outputId": "6d01c5d4-97a0-4c46-c28b-15b219f98d2e"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 36
        }
      ],
      "source": [
        "%%sql\n",
        "-- Show unique cleaned categories in alphabetical order.\n",
        "\n",
        "\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "32f1a5d4",
      "metadata": {
        "id": "32f1a5d4"
      },
      "source": [
        "## Part D1 — Case matters"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": 37,
      "id": "db9d0815",
      "metadata": {
        "id": "db9d0815",
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "outputId": "5b1b7571-3836-4279-ef3b-4d2e82a19d5b"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 37
        }
      ],
      "source": [
        "%%sql\n",
        "-- Return rows where category = 'apple' (exact match).\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "8c2cca40",
      "metadata": {
        "id": "8c2cca40"
      },
      "source": [
        "## Part D2 — Case-insensitive match"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": 38,
      "id": "049ac8b6",
      "metadata": {
        "id": "049ac8b6",
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "outputId": "f593a149-c4ba-4ded-a872-bc23c85796ca"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 38
        }
      ],
      "source": [
        "%%sql\n",
        "-- Return rows that are apple regardless of case/spaces using LOWER(TRIM(category)) = 'apple'.\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "3d0614bd",
      "metadata": {
        "id": "3d0614bd"
      },
      "source": [
        "## Part D3 — Empty strings aren’t NULL"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": 39,
      "id": "1acf51d6",
      "metadata": {
        "id": "1acf51d6",
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "outputId": "0a0d9118-4e94-4fdb-92ea-13abd2546f4d"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 39
        }
      ],
      "source": [
        "%%sql\n",
        "-- Show rows where category is an empty string '' (NOT NULL).\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "8915f947",
      "metadata": {
        "id": "8915f947"
      },
      "source": [
        "## Part D4 — Find NULL categories"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": 40,
      "id": "04a68c98",
      "metadata": {
        "id": "04a68c98",
        "colab": {
          "base_uri": "https://localhost:8080/"
        },
        "outputId": "d7b3431c-6c43-4831-9c6a-0de5fe984d4c"
      },
      "outputs": [
        {
          "output_type": "stream",
          "name": "stdout",
          "text": [
            " * sqlite:///text_issues.db\n",
            "Done.\n"
          ]
        },
        {
          "output_type": "execute_result",
          "data": {
            "text/plain": [
              "[]"
            ]
          },
          "metadata": {},
          "execution_count": 40
        }
      ],
      "source": [
        "%%sql\n",
        "-- Show rows where category IS NULL.\n",
        "\n",
        "\n"
      ]
    }
  ],
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "display_name": "Python 3",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.x"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'id': '063ed243',
   'metadata': {'id': '063ed243'},
   'source': ['# IC (Feb 04) — Text Data in SQL (Student Version)\n',
    '\n',
    'This in-class exercise practices the ideas from the Feb 04 slides:\n',
    '- Text / character data and why it’s tricky\n',
    '- `GROUP BY` + `COUNT()`\n',
    '- `ORDER BY` (frequency vs alphabetical)\n',
    '- Case sensitivity and `LOWER()` / `UPPER()`\n',
    '- Spaces and `TRIM()`\n',
    '- `LIKE` searches (and why they can be surprising)\n',
    '\n',
    '**Instructions:** Run cells top → bottom. Write SQL inside the `query` blocks.\n']},
  {'cell_type': 'markdown',
   'id': '4577a25d',
   'metadata': {'id': '4577a25d'},
   'source': ['## 0) Setup (Run)\n',
    'Creates a SQLite database `text_issues.db` and a table `product` with messy text.']},
  {'cell_type': 'code',
   'source': ['!pip install --upgrade pandas ipython-sql prettytable==3.10.1'],
   'metadata': {'colab': {'base_uri': 'https://localh